# ============================================================
# Kaggle Notebook: Graph-DKT 10k retrain — SELF-RESUMING variant
# ============================================================
# STATE: DKT 5/5 folds DONE (leak-fixed retrain, best NewAUC 0.7029).
#        Graph-DKT folds 0-1 done; folds 2-4 remaining as of 2026-11-07.
#
# HOW TO RUN (no cell editing needed — ever):
#   1. Attach BOTH datasets: training.csv  AND  gdkt10k-folds (carry dataset).
#   2. GPU T4 x1 (or x2), Internet ON. File -> Save & Run All (Commit).
#   3. The resume cell md5-compares weights against the git-carried old ensemble
#      and trains exactly the unproven folds (max 2 per 12h commit).
#   4. If the session dies mid-fold: click Save & Run All again — it recomputes.
#   5. When the carry dataset gains folds 2-3, run again -> last fold + eval + zip.
#      Eval stdout MUST be copied out of the log — results.zip is weights-only.
#
# CARRY-DATASET RULE: upload only VERIFIED completed fold weights (the
# sequential-successor check cannot detect an interrupted best-so-far fold).
# The gdkt10k-folds dataset currently holds: dkt_10k_fold0-4 + graph_dkt_10k_fold0-1.
# ============================================================


### CELL 0 — Install dependencies


In [ ]:
# Install torch-geometric so the Graph-DKT path is activated.
# The dense GCN uses pure PyTorch (einsum) under the hood, so heavy binary dependencies
# (like pyg_lib, torch_scatter, torch_sparse, torch_cluster) are NOT required.
# Kaggle already has a fully-functional PyTorch + CUDA pre-installed.
# Note: You MUST toggle 'Internet on' in the right panel under Settings to download packages!
try:
    import torch_geometric
    print("torch_geometric is already installed!")
except ImportError:
    print("Installing torch-geometric...")
    !pip install torch-geometric -q || echo "WARNING: Could not install torch-geometric. Check if Internet is toggled ON in the right panel under Settings. Falling back to pure PyTorch dense GCN mode."

!pip install scikit-learn pandas tqdm -q

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    try:
        print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    except AttributeError:
        props = torch.cuda.get_device_properties(0)
        mem = getattr(props, 'total_memory', getattr(props, 'total_mem', 0))
        print(f"VRAM: {mem / 1e9:.1f} GB")

### CELL 1 — Copy training.csv from Kaggle input


In [ ]:
# When you use Kaggle's "Add Data" → "Upload", the file
# lands in /kaggle/input/<dataset-name>/training.csv

import os, shutil

# Find training.csv in Kaggle input directory
input_dir = "/kaggle/input"
training_csv = None
for root, dirs, files in os.walk(input_dir):
    if "training.csv" in files:
        training_csv = os.path.join(root, "training.csv")
        break

if training_csv is None:
    print("ERROR: training.csv not found in Kaggle input!")
    print("Go to: Add data → Upload → select training.csv")
    print("Then re-run this cell.")
    import sys; sys.exit(1)

print(f"Found: {training_csv}")
print(f"Size: {os.path.getsize(training_csv) / 1e6:.1f} MB")

### CELL 2 — Clone repo and copy training data


In [ ]:
!git clone https://github.com/MannPatel1236/CP-Coach.git /kaggle/working/CP-Coach

# Copy training.csv into repo
data_dir = "/kaggle/working/CP-Coach/backend/data"
os.makedirs(data_dir, exist_ok=True)
shutil.copy(training_csv, os.path.join(data_dir, "training.csv"))

print(f"Copied training.csv to {data_dir}")
!wc -l /kaggle/working/CP-Coach/backend/data/training.csv

### CELL 3 — Verify data + quick stats


In [ ]:
%cd /kaggle/working/CP-Coach/backend

import sys
sys.path.insert(0, "/kaggle/working/CP-Coach/backend")

from data.topic_graph import CPTopicGraph
from training.train_dkt import load_csv_sequences

tg = CPTopicGraph()
# Let's load with max_seq_len=10000 matching our training configurations
seqs = load_csv_sequences("data/training.csv", tg, max_seq_len=10000)
print(f"Loaded {len(seqs)} user sequences (max_seq_len=10000)")

import numpy as np
lengths = np.array([len(s) for s in seqs])
print(f"Sequence length stats (after truncation):")
for p in [50, 75, 90, 95, 99, 100]:
    print(f"  p{p}: {np.percentile(lengths, p):.0f}")

### CELL 4 — DKT smoke test (3 epochs, 1 fold)


In [ ]:
# SMOKE TEST — skipped in the resume variant: the 10k config is already proven on
# GPU by the commit-1/2 runs (no OOM), and each resume commit re-saves time.
print("Smoke test skipped (config unchanged and already proven at seq_len=10000).")

### CELL 5 — DKT full retrain (5 folds × 50 epochs)


In [ ]:
# DKT RETRAIN — COMPLETE (5/5 folds, best fold NewAUC 0.7029). Weights ride in the
# carry dataset (dkt_10k_fold0-4) and are copied into weights/ by the resume cell.
# This cell is intentionally inert; kept only so the notebook order stays stable.
print("DKT training already complete — weights provided by the carry dataset.")

### CELL 7 — Resume step 1/3: fingerprint the git-carried old ensemble

In [ ]:
# RESUME STEP 1 — snapshot the git-carried OLD ensemble md5s BEFORE anything overwrites them.
# The repo clone force-adds the OLD leak-era graph_dkt_10k_fold{0..4}.pt; these checksums
# are the "old fingerprint" used to tell retrained folds from carry-through ghosts.
import hashlib, json, os
def _md5(path):
    h = hashlib.md5()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

old_md5 = {}
for i in range(5):
    p = f"weights/graph_dkt_10k_fold{i}.pt"
    old_md5[i] = _md5(p) if os.path.exists(p) else None
json.dump(old_md5, open("/kaggle/working/old_md5.json", "w"))
print("Old-ensemble fingerprints:", json.dumps(old_md5, indent=2))

### CELL 7b — Resume step 2/3: copy verified weights from the carry dataset

In [ ]:
# RESUME STEP 2 — copy verified fold weights from the carry dataset (gdkt10k-folds).
# Anything copied here is treated as PROVEN complete; folds 3-4 of the old ensemble
# are NOT in the dataset and will be retrained over the git-carried copies.
import json, os, shutil
trusted = []
for root, _, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".pt") and "_10k_fold" in f:
            shutil.copy(os.path.join(root, f), os.path.join("weights", f))
            trusted.append(f)
json.dump(sorted(trusted), open("/kaggle/working/trusted.json", "w"))
print(f"Copied {len(trusted)} carry files:", sorted(trusted))
if not any("_10k_fold" in f for f in trusted):
    print("WARNING: no *_10k_fold*.pt found in input — attach the gdkt10k-folds dataset!")

### CELL 7c — Resume step 3/3: compute the unproven fold range

In [ ]:
# RESUME STEP 3 — decide what still needs training.
# PROVEN = came from the carry dataset, OR has a later fold whose weights differ
# from the old fingerprint (sequential training: reaching fold g means fold f done).
import json, glob
old_md5 = {int(k): v for k, v in json.load(open("/kaggle/working/old_md5.json")).items()}
trusted_gdkt = {int(f.rsplit("fold", 1)[1].split(".")[0]) for f in json.load(open("/kaggle/working/trusted.json"))
                if f.startswith("graph_dkt_10k_fold")}

import hashlib
def _md5(path):
    h = hashlib.md5()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

current = {}
for i in range(5):
    p = f"weights/graph_dkt_10k_fold{i}.pt"
    if os.path.exists(p):
        current[i] = _md5(p)

proven = set(trusted_gdkt)
for i in range(5):
    if i in current and current[i] != old_md5.get(i) and any(
            g in current and current[g] != old_md5.get(g) for g in range(i + 1, 5)):
        proven.add(i)  # a later fold started => this fold finished

RESUME_START = next((i for i in range(5) if i not in proven), 5)
RESUME_END = min(RESUME_START + 1, 4)  # max 2 folds per 12h commit (~210s/epoch measured)
import os
os.environ["RESUME_START"], os.environ["RESUME_END"] = str(RESUME_START), str(RESUME_END)
print(f"Proven folds: {sorted(proven)}")
print(f"Plan: train folds {RESUME_START}..{RESUME_END}" if RESUME_START <= 4
      else "Plan: all 5 folds proven — skip training, go to eval")

### CELL 8 — Graph-DKT retrain (unproven folds only)

In [ ]:
# Graph-DKT training — only the unproven fold range. (--start-fold/--end-fold are
# 0-indexed, inclusive; each fold is an independent model, so this is lossless.)
import os, subprocess
start, end = int(os.environ["RESUME_START"]), int(os.environ["RESUME_END"])
if start <= 4:
    cmd = ["python3", "-m", "training.train_dkt", "--data", "data/training.csv",
           "--model", "graph_dkt", "--epochs", "50", "--folds", "5",
           "--out", "weights/graph_dkt_10k.pt", "--batch", "16", "--device", "cuda",
           "--max-seq-len", "10000", "--start-fold", str(start), "--end-fold", str(end)]
    print("Running:", " ".join(cmd))
    r = subprocess.run(cmd)
    if r.returncode != 0:
        raise RuntimeError(f"training failed with exit code {r.returncode}")
else:
    print("All graph_dkt folds already trained — skipping training cell.")

### CELL 9 — Compare results (gated: final commit or eval-only re-run)

In [ ]:
# Eval runs only when all 5 graph_dkt folds are new (last commit, or a re-run of one).
# SAVE THIS OUTPUT — results.zip is weights-only; this stdout is the paper's honest
# AUC + P@K table (pooled ~0.72 expected post-leak-fix, NOT a regression; NewAUC ~0.69+).
import os, subprocess
start, end = int(os.environ["RESUME_START"]), int(os.environ["RESUME_END"])
if start > 4 or end == 4:
    cmd = ["python3", "-m", "training.eval_folds", "--segment-by-length", "--device", "cpu",
           "--max-seq-len", "10000", "--dkt-weights", "dkt_10k_fold{fold}.pt",
           "--gdkt-weights", "graph_dkt_10k_fold{fold}.pt"]
    r = subprocess.run(cmd)
    if r.returncode != 0:
        raise RuntimeError(f"eval failed with exit code {r.returncode}")
    print("\n>>> COPY THE TABLES ABOVE OUT OF THIS LOG — results.zip is weights-only <<<")
else:
    print(f"Folds {end + 1}..4 still untrained — skipping eval until the final commit.")

### CELL 10 — Download results (gated)

In [ ]:
# Zip weights (results.zip is weights-only by design — Kaggle output packaging).
import zipfile, glob, os
start, end = int(os.environ["RESUME_START"]), int(os.environ["RESUME_END"])
if start > 4 or end == 4:
    with zipfile.ZipFile("/kaggle/working/results.zip", "w") as z:
        for f in sorted(glob.glob("weights/*.pt")):
            z.write(f)
        print("Packed weights:")
        for name in z.namelist():
            print(f"  {name}")
    print("\nDone! Download results.zip from the Output panel.")
else:
    print("Intermediate commit — fold weights persist in this version's output; no zip needed.")
